[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miguepoloc/toma-decisiones-mcda/blob/main/01_ahp_iot_palmor.ipynb)

# AHP, caso IoT/WSN Palmor

Elección de tecnología de comunicación (**LoRaWAN, GSM/GPRS, Sigfox, Zigbee**) para una red de sensores IoT/WSN de monitoreo agroclimático en Palmor, corregimiento de Ciénaga (Sierra Nevada de Santa Marta, Magdalena), zona con conectividad limitada verificada vía MinTIC. Los 4 criterios (Alcance de comunicación, Autonomía de batería, Infraestructura/cobertura comercial en Colombia, Madurez/viabilidad comercial del proveedor) salieron del panel de evidencia de la Sesión 1. Los valores técnicos de la matriz de decisión son reales, verificados vía WebSearch (datasheets SIMCom/DigiKey, The Things Network, Lauridsen et al. 2019 *Sensors*/MDPI, noticias de apagado de 2G en Colombia). Contenido completo y verificado en la Sesión 2 (AHP) del curso.

In [1]:
!pip install -q pyDecision


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
from pyDecision.algorithm import ahp_method

tecnologias = ["LoRaWAN", "GSM/GPRS", "Sigfox", "Zigbee"]
criterios = ["Alcance", "Autonomia", "Infraestructura", "Madurez"]

def gmean(m1, m2, m3):
    m1, m2, m3 = np.array(m1, dtype=float), np.array(m2, dtype=float), np.array(m3, dtype=float)
    return (m1 * m2 * m3) ** (1 / 3)

## Paso 1, matriz de criterios

Panel de 3 expertos (mismo panel en las 5 matrices de este notebook, Forman & Peniwati 1998: media geométrica de juicios individuales). Juicios de Saaty de cada experto:

In [3]:
m_criterios = gmean(
    [[1, 2, 1/3, 2], [1/2, 1, 1/4, 1], [3, 4, 1, 4], [1/2, 1, 1/4, 1]],
    [[1, 1/2, 1/4, 2], [2, 1, 1/2, 4], [4, 2, 1, 5], [1/2, 1/4, 1/5, 1]],
    [[1, 1/3, 1/3, 1], [3, 1, 1, 5], [3, 1, 1, 4], [1, 1/5, 1/4, 1]],
)
w_criterios, cr_criterios = ahp_method(m_criterios, wd='m')
print("Pesos de criterios:", dict(zip(criterios, np.round(w_criterios, 4))))
print("CR:", round(cr_criterios, 4))

Pesos de criterios: {'Alcance': np.float64(0.1604), 'Autonomia': np.float64(0.2502), 'Infraestructura': np.float64(0.4875), 'Madurez': np.float64(0.1019)}
CR: 0.0032


## Paso 2, una matriz por criterio, comparando las 4 tecnologías

In [4]:
m_alcance = gmean(
    [[1, 1, 1/3, 5], [1, 1, 1/3, 5], [3, 3, 1, 9], [1/5, 1/5, 1/9, 1]],
    [[1, 1, 1/2, 3], [1, 1, 1/2, 3], [2, 2, 1, 6], [1/3, 1/3, 1/6, 1]],
    [[1, 2, 1/2, 4], [1/2, 1, 1/3, 3], [2, 3, 1, 7], [1/4, 1/3, 1/7, 1]],
)
m_autonomia = gmean(
    [[1, 5, 2, 3], [1/5, 1, 1/2, 1/2], [1/2, 2, 1, 1], [1/3, 2, 1, 1]],
    [[1, 7, 3, 4], [1/7, 1, 1/3, 1/3], [1/3, 3, 1, 1], [1/4, 3, 1, 1]],
    [[1, 9, 4, 5], [1/9, 1, 1/4, 1/4], [1/4, 4, 1, 1], [1/5, 4, 1, 1]],
)
m_infraestructura = gmean(
    [[1, 1/4, 1/8, 1], [4, 1, 1/2, 4], [8, 2, 1, 8], [1, 1/4, 1/8, 1]],
    [[1, 1/2, 1/4, 1], [2, 1, 1/2, 2], [4, 2, 1, 4], [1, 1/2, 1/4, 1]],
    [[1, 1/3, 1/7, 1], [3, 1, 1/2, 3], [7, 2, 1, 6], [1, 1/3, 1/6, 1]],
)
m_madurez = gmean(
    [[1, 3, 9, 1], [1/3, 1, 3, 1/3], [1/9, 1/3, 1, 1/9], [1, 3, 9, 1]],
    [[1, 2, 6, 1], [1/2, 1, 2, 1/2], [1/6, 1/2, 1, 1/6], [1, 2, 6, 1]],
    [[1, 2, 4, 1], [1/2, 1, 1, 1/2], [1/4, 1, 1, 1/4], [1, 2, 4, 1]],
)

w_alcance, cr_alcance = ahp_method(m_alcance, wd='m')
w_autonomia, cr_autonomia = ahp_method(m_autonomia, wd='m')
w_infraestructura, cr_infraestructura = ahp_method(m_infraestructura, wd='m')
w_madurez, cr_madurez = ahp_method(m_madurez, wd='m')

for nombre, w, cr in [("Alcance", w_alcance, cr_alcance), ("Autonomia", w_autonomia, cr_autonomia),
                      ("Infraestructura", w_infraestructura, cr_infraestructura), ("Madurez", w_madurez, cr_madurez)]:
    print(f"{nombre:16s}", dict(zip(tecnologias, np.round(w, 4))), "CR=", round(cr, 4))

Alcance          {'LoRaWAN': np.float64(0.2368), 'GSM/GPRS': np.float64(0.1996), 'Sigfox': np.float64(0.5017), 'Zigbee': np.float64(0.0619)} CR= 0.0032
Autonomia        {'LoRaWAN': np.float64(0.5624), 'GSM/GPRS': np.float64(0.0698), 'Sigfox': np.float64(0.1904), 'Zigbee': np.float64(0.1774)} CR= 0.0091
Infraestructura  {'LoRaWAN': np.float64(0.0921), 'GSM/GPRS': np.float64(0.2692), 'Sigfox': np.float64(0.5453), 'Zigbee': np.float64(0.0933)} CR= 0.0001
Madurez          {'LoRaWAN': np.float64(0.3867), 'GSM/GPRS': np.float64(0.1553), 'Sigfox': np.float64(0.0713), 'Zigbee': np.float64(0.3867)} CR= 0.0062


## Paso 3, síntesis global

Prioridad global de cada tecnología = suma ponderada de sus prioridades locales por el peso de cada criterio.

In [5]:
prioridad_local = np.column_stack([w_alcance, w_autonomia, w_infraestructura, w_madurez])
prioridad_global = prioridad_local @ w_criterios

print("Ranking final AHP:")
for t, p in sorted(zip(tecnologias, prioridad_global), key=lambda x: -x[1]):
    print(f"  {t:10s} {p:.4f}")

Ranking final AHP:
  Sigfox     0.4012
  LoRaWAN    0.2630
  GSM/GPRS   0.1966
  Zigbee     0.1392


**Resultado esperado** (coincide con lo publicado en la Sesión 2 del curso): 1º Sigfox (0.4012) · 2º LoRaWAN (0.2630) · 3º GSM/GPRS (0.1966) · 4º Zigbee (0.1392).